In [1]:
pip install pygame numpy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


Exercício 1:

In [ ]:
import pygame
import math
import sys

# --- Cálculos Teóricos (Analíticos) ---
# T1: t1=4s, v=0.5 m/s, w=0
dx1 = 0.5 * 4.0
dy1 = 0.0
th1 = 0.0

# T2: t2=2s, v=0, w=pi/4 rad/s
th2 = th1 + (math.pi / 4.0) * 2.0  # pi/2 rad (90°)

# T3: t3=3s, v=0.4 m/s, w=0 (direção th2)
dx3 = 0.4 * 3.0 * math.cos(th2)
dy3 = 0.4 * 3.0 * math.sin(th2)
th_final_teo = th2

x_teo = dx1 + dx3
y_teo = dy1 + dy3

print("=== POSE TEÓRICA ESPERADA ===")
print(f"X: {x_teo:.4f} m | Y: {y_teo:.4f} m | θ: {th_final_teo:.4f} rad ({math.degrees(th_final_teo):.2f}°)\n")

# --- Simulação Gráfica ---
pygame.init()
WIDTH, HEIGHT = 800, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Lab 1: Validador de Pose em Malha Aberta")
clock = pygame.time.Clock()
font = pygame.font.SysFont("monospace", 15)

SCALE = 80.0  # 1 metro = 80 pixels
ORIGIN = (100, 450)

# Estado do robô (SI)
x, y, theta = 0.0, 0.0, 0.0
trajectory = []

t_total = 0.0
running = True

while running:
    dt = clock.tick(60) / 1000.0  # segundos por frame
    t_total += dt

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Controle em malha aberta temporizado
    if t_total <= 4.0:
        v, w = 0.5, 0.0
    elif t_total <= 6.0:
        v, w = 0.0, math.pi / 4.0
    elif t_total <= 9.0:
        v, w = 0.4, 0.0
    else:
        v, w = 0.0, 0.0

    # Cinemática Diferencial (Integração de Euler)
    theta += w * dt
    x += v * math.cos(theta) * dt
    y += v * math.sin(theta) * dt

    # Converte coordenadas cartesianas para a tela (y invertido no Pygame)
    px = ORIGIN[0] + int(x * SCALE)
    py = ORIGIN[1] - int(y * SCALE)
    trajectory.append((px, py))

    # Desenho
    screen.fill((25, 25, 30))
    
    # Trajetória
    if len(trajectory) > 1:
        pygame.draw.lines(screen, (80, 180, 255), False, trajectory, 2)

    # Robô
    pygame.draw.circle(screen, (240, 70, 70), (px, py), 12)
    hx = px + int(18 * math.cos(theta))
    hy = py - int(18 * math.sin(theta))
    pygame.draw.line(screen, (255, 255, 255), (px, py), (hx, hy), 3)

    # Textos
    txt1 = font.render(f"Tempo: {t_total:.2f}s | v: {v:.2f} m/s | w: {w:.2f} rad/s", True, (220, 220, 220))
    txt2 = font.render(f"Simulado -> x: {x:.3f} m | y: {y:.3f} m | theta: {math.degrees(theta):.1f}°", True, (100, 255, 100))
    txt3 = font.render(f"Teorico  -> x: {x_teo:.3f} m | y: {y_teo:.3f} m | theta: {math.degrees(th_final_teo):.1f}°", True, (255, 215, 0))
    screen.blit(txt1, (20, 20))
    screen.blit(txt2, (20, 45))
    screen.blit(txt3, (20, 70))

    pygame.display.flip()

print("=== POSE FINAL SIMULADA ===")
print(f"X: {x:.4f} m | Y: {y:.4f} m | θ: {theta:.4f} rad ({math.degrees(theta):.2f}°)")
pygame.quit()
sys.exit()



Exercício 2:

In [ ]:
import pygame
import math
import sys

pygame.init()
WIDTH, HEIGHT = 900, 700
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Lab 2: Calculadora de Giro Ackermann")
clock = pygame.time.Clock()
font = pygame.font.SysFont("monospace", 15)

L = 2.0              # Entre-eixos (m)
MAX_STEER = math.radians(30.0)  # ±30°
SCALE = 20.0         # 1 metro = 20 pixels

x, y = WIDTH // 2, HEIGHT // 2
theta = 0.0
v = 4.0              # m/s
phi = 0.0            # rad

trail = []
running = True

while running:
    dt = clock.tick(60) / 1000.0

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif event.type == pygame.KEYDOWN and event.key == pygame.K_SPACE:
            trail.clear()

    keys = pygame.key.get_pressed()
    if keys[pygame.K_UP]:    v += 2.0 * dt
    if keys[pygame.K_DOWN]:  v = max(0.0, v - 2.0 * dt)
    if keys[pygame.K_LEFT]:  phi = min(MAX_STEER, phi + 0.8 * dt)
    if keys[pygame.K_RIGHT]: phi = max(-MAX_STEER, phi - 0.8 * dt)

    # Relação Cinemática de Ackermann
    if abs(phi) > 1e-4:
        omega = (v / L) * math.tan(phi)
        R = L / math.tan(phi)
    else:
        omega = 0.0
        R = float('inf')

    # Integração
    theta += omega * dt
    x += v * math.cos(theta) * SCALE * dt
    y -= v * math.sin(theta) * SCALE * dt

    trail.append((int(x), int(y)))
    if len(trail) > 1200:
        trail.pop(0)

    screen.fill((20, 22, 28))

    if len(trail) > 1:
        pygame.draw.lines(screen, (245, 170, 50), False, trail, 2)

    # Desenho do veículo Ackermann (retângulo orientado)
    car_len, car_wid = int(L * SCALE), int(L * SCALE * 0.5)
    surf = pygame.Surface((car_len, car_wid), pygame.SRCALPHA)
    pygame.draw.rect(surf, (200, 60, 60), (0, 0, car_len, car_wid), border_radius=4)
    # Rodas dianteiras esterçadas
    wheel_w, wheel_h = 10, 4
    pygame.draw.line(surf, (255, 255, 255), (car_len - 5, car_wid // 2), 
                     (car_len - 5 + int(8 * math.cos(phi)), car_wid // 2 - int(8 * math.sin(phi))), 3)
    rotated = pygame.transform.rotate(surf, math.degrees(theta))
    screen.blit(rotated, rotated.get_rect(center=(int(x), int(y))))

    # Painel de Telemetria
    r_str = f"{R:.2f} m" if R != float('inf') else "Infinito (Linha reta)"
    t1 = font.render(f"Velocidade (v): {v:.2f} m/s | Angulo esterco (phi): {math.degrees(phi):.1f}° (Max: ±30°)", True, (255, 255, 255))
    t2 = font.render(f"Velocidade Angular (w): {omega:.3f} rad/s", True, (100, 220, 255))
    t3 = font.render(f"Raio de Curva (R = L/tan(phi)): {r_str}", True, (255, 215, 0))
    t4 = font.render("Controles: Setas Cima/Baixo (v), Esquerda/Direita (esterco), Espaco (limpar)", True, (160, 160, 160))

    screen.blit(t1, (20, 20))
    screen.blit(t2, (20, 45))
    screen.blit(t3, (20, 70))
    screen.blit(t4, (20, 100))

    pygame.display.flip()

pygame.quit()
sys.exit()


Exercício 3: 

In [1]:
import pygame
import math
import numpy as np
import sys

pygame.init()
WIDTH, HEIGHT = 900, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Lab 3: Varredura Sensorial com Filtro de Alcance")
clock = pygame.time.Clock()
font = pygame.font.SysFont("monospace", 14)

ROBOT_POS = (250, 350)
ROBOT_ANGLE = -math.pi / 2  # apontando para cima
NUM_RAYS = 7
MAX_RANGE = 200.0
MIN_THRESHOLD = 10.0

angles = np.linspace(-math.pi / 2, math.pi / 2, NUM_RAYS)
running = True

while running:
    clock.tick(30)
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    mouse_x, mouse_y = pygame.mouse.get_pos()
    obstacle_pos = np.array([mouse_x, mouse_y])

    screen.fill((18, 18, 24))

    # Obstáculo móvel
    pygame.draw.circle(screen, (220, 50, 50), (mouse_x, mouse_y), 25)

    # Robô
    pygame.draw.circle(screen, (50, 150, 250), ROBOT_POS, 15)

    panel_y = 40
    screen.blit(font.render("FEIXE | D_REAL  | D_RUIDOSA | D_FILTRADA (STATUS)", True, (255, 215, 0)), (480, 15))

    for i, alpha in enumerate(angles):
        ray_angle = ROBOT_ANGLE + alpha
        ray_dir = np.array([math.cos(ray_angle), math.sin(ray_angle)])

        # Cálculo de distância geométrica simplificada (raio contra o círculo obstáculo)
        v_to_obs = obstacle_pos - np.array(ROBOT_POS)
        proj = np.dot(v_to_obs, ray_dir)
        d_real = MAX_RANGE

        if proj > 0:
            perp_dist = np.linalg.norm(v_to_obs - proj * ray_dir)
            if perp_dist <= 25.0:  # colidiu com raio do obstáculo
                d_real = min(MAX_RANGE, max(0.0, proj - math.sqrt(25.0**2 - perp_dist**2)))

        # 1. Ruído Gaussiano
        d_noise = d_real + np.random.normal(0.0, 5.0)

        # 2. Filtro de Limiar (Threshold)
        if d_noise < MIN_THRESHOLD:
            d_filtered = None
            status = "DESCARTE (<10px)"
        elif d_noise > MAX_RANGE:
            d_filtered = MAX_RANGE
            status = "CLAMPADO (200px)"
        else:
            d_filtered = d_noise
            status = f"{d_filtered:.1f} px"

        # Desenho do raio
        draw_dist = d_filtered if d_filtered is not None else MAX_RANGE
        end_pt = (ROBOT_POS[0] + int(draw_dist * ray_dir[0]), 
                  ROBOT_POS[1] + int(draw_dist * ray_dir[1]))
        
        color = (120, 120, 120) if d_filtered is None else (0, 255, 120)
        pygame.draw.line(screen, color, ROBOT_POS, end_pt, 2)
        pygame.draw.circle(screen, (255, 255, 255), end_pt, 3)

        # Painel lateral
        txt = font.render(f"F{i+1}:   {d_real:6.1f} | {d_noise:6.1f}   | {status}", True, (200, 200, 200))
        screen.blit(txt, (480, panel_y))
        panel_y += 24

    screen.blit(font.render("Mova o mouse para posicionar o obstáculo", True, (150, 150, 150)), (20, 20))
    pygame.display.flip()

pygame.quit()
sys.exit()


pygame 2.6.1 (SDL 2.28.4, Python 3.10.12)
Hello from the pygame community. https://www.pygame.org/contribute.html


KeyboardInterrupt: 

Exercício 4:

In [2]:
import pygame
import math
import sys

pygame.init()
WIDTH, HEIGHT = 800, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Lab 4: Veículo Braitenberg - Agressão / Atração")
clock = pygame.time.Clock()
font = pygame.font.SysFont("monospace", 14)

def reset_robot():
    return 150.0, 300.0, 0.0

x, y, theta = reset_robot()
v0 = 40.0
alpha = 70.0
d_max = 220.0
wheel_base = 26.0

obs_pos = (550, 300)
obs_radius = 45

running = True
trail = []

while running:
    dt = clock.tick(60) / 1000.0

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif event.type == pygame.KEYDOWN and event.key == pygame.K_r:
            x, y, theta = reset_robot()
            trail.clear()

    # Sensores posicionados a ±35° do nariz do robô
    s_ang_l = theta - math.radians(35)
    s_ang_r = theta + math.radians(35)
    
    pos_l = (x + 16 * math.cos(s_ang_l), y + 16 * math.sin(s_ang_l))
    pos_r = (x + 16 * math.cos(s_ang_r), y + 16 * math.sin(s_ang_r))

    dist_l = math.hypot(obs_pos[0] - pos_l[0], obs_pos[1] - pos_l[1]) - obs_radius
    dist_r = math.hypot(obs_pos[0] - pos_r[0], obs_pos[1] - pos_r[1]) - obs_radius

    d_esq = max(0.0, min(d_max, dist_l))
    d_dir = max(0.0, min(d_max, dist_r))

    # Conexões Diretas (Não-cruzadas)
    vL = v0 + alpha * (1.0 - (d_esq / d_max))
    vR = v0 + alpha * (1.0 - (d_dir / d_max))

    # Cinemática diferencial
    v_lin = (vR + vL) / 2.0
    omega = (vR - vL) / wheel_base

    theta += omega * dt
    x += v_lin * math.cos(theta) * dt
    y += v_lin * math.sin(theta) * dt

    trail.append((int(x), int(y)))

    # Renderização
    screen.fill((22, 22, 28))
    
    if len(trail) > 1:
        pygame.draw.lines(screen, (255, 90, 90), False, trail, 2)

    # Obstáculo
    pygame.draw.circle(screen, (220, 60, 60), obs_pos, obs_radius)
    pygame.draw.circle(screen, (255, 255, 255), obs_pos, obs_radius, 2)

    # Linhas dos sensores
    pygame.draw.line(screen, (255, 200, 0), pos_l, (pos_l[0] + d_esq * math.cos(s_ang_l), pos_l[1] + d_esq * math.sin(s_ang_l)), 1)
    pygame.draw.line(screen, (0, 200, 255), pos_r, (pos_r[0] + d_dir * math.cos(s_ang_r), pos_r[1] + d_dir * math.sin(s_ang_r)), 1)

    # Robô
    pygame.draw.circle(screen, (60, 160, 240), (int(x), int(y)), 15)
    pygame.draw.line(screen, (255, 255, 255), (int(x), int(y)), 
                     (int(x + 20 * math.cos(theta)), int(y + 20 * math.sin(theta))), 3)

    screen.blit(font.render(f"Sensor Esq: {d_esq:.1f}px -> vL: {vL:.1f}", True, (255, 200, 0)), (20, 20))
    screen.blit(font.render(f"Sensor Dir: {d_dir:.1f}px -> vR: {vR:.1f}", True, (0, 200, 255)), (20, 45))
    screen.blit(font.render("Comportamento: Agressão / Atração (Aperte R para reiniciar)", True, (200, 200, 200)), (20, 70))

    pygame.display.flip()

pygame.quit()
sys.exit()



KeyboardInterrupt: 

Exercício 5:

In [3]:
import pygame
import math
import sys

pygame.init()
WIDTH, HEIGHT = 900, 500
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Lab 5: Centralização Autônoma em Corredor (Controle P)")
clock = pygame.time.Clock()
font = pygame.font.SysFont("monospace", 14)

WALL_TOP_Y = 150
WALL_BOT_Y = 350
CORRIDOR_CENTER_Y = (WALL_TOP_Y + WALL_BOT_Y) / 2.0  # 250 px

# Estado inicial (desalinhado intencionalmente)
x = 50.0
y = 190.0  # mais próximo da parede superior
theta = 0.0

v = 40.0       # px/s
Kp = 0.01

trail = []
running = True

while running:
    dt = clock.tick(60) / 1000.0

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Perturbação manual de teste
    keys = pygame.key.get_pressed()
    if keys[pygame.K_UP]:   y -= 40.0 * dt
    if keys[pygame.K_DOWN]: y += 40.0 * dt

    # Sensores laterais a -90° (esquerda) e +90° (direita) em relação à orientação
    d_esq = max(0.0, y - WALL_TOP_Y)
    d_dir = max(0.0, WALL_BOT_Y - y)

    # Lei de Controle Proporcional
    # No sistema de eixos de tela do Pygame (y cresce para baixo),
    # se d_esq < d_dir (mais perto do topo), o erro e = d_esq - d_dir é negativo.
    # Para girar no sentido horário (descendo em y), precisamos que w seja positivo:
    erro = d_esq - d_dir
    omega = -Kp * erro

    # Integração cinemática
    theta += omega * dt
    x += v * math.cos(theta) * dt
    y += v * math.sin(theta) * dt

    # Loop da arena
    if x > WIDTH - 40:
        x = 50.0
        trail.clear()

    trail.append((int(x), int(y)))
    if len(trail) > 600:
        trail.pop(0)

    # Renderização
    screen.fill((20, 20, 26))

    # Paredes e centro de referência
    pygame.draw.line(screen, (100, 100, 100), (0, WALL_TOP_Y), (WIDTH, WALL_TOP_Y), 6)
    pygame.draw.line(screen, (100, 100, 100), (0, WALL_BOT_Y), (WIDTH, WALL_BOT_Y), 6)
    pygame.draw.line(screen, (50, 60, 70), (0, int(CORRIDOR_CENTER_Y)), (WIDTH, int(CORRIDOR_CENTER_Y)), 1)

    # Trajetória
    if len(trail) > 1:
        pygame.draw.lines(screen, (50, 200, 100), False, trail, 2)

    # Feixes dos sensores
    pygame.draw.line(screen, (255, 100, 100), (int(x), int(y)), (int(x), WALL_TOP_Y), 2)
    pygame.draw.line(screen, (100, 100, 255), (int(x), int(y)), (int(x), WALL_BOT_Y), 2)

    # Robô
    pygame.draw.circle(screen, (240, 240, 240), (int(x), int(y)), 12)
    hx = int(x + 18 * math.cos(theta))
    hy = int(y + 18 * math.sin(theta))
    pygame.draw.line(screen, (255, 50, 50), (int(x), int(y)), (hx, hy), 3)

    # Telemetria
    screen.blit(font.render(f"d_esq: {d_esq:.1f} px | d_dir: {d_dir:.1f} px | Erro: {erro:.1f} px", True, (255, 255, 255)), (20, 20))
    screen.blit(font.render(f"w: {omega:.4f} rad/s (Kp = {Kp}) | v = {v:.1f} px/s", True, (100, 255, 150)), (20, 45))
    screen.blit(font.render("Setas Cima/Baixo: aplicar perturbacao no robô", True, (150, 150, 150)), (20, 70))

    pygame.display.flip()

pygame.quit()
sys.exit()


SystemExit: 

/usr/lib/python3/dist-packages/IPython/core/interactiveshell.py:3465: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
